# Módulo 2 (a) · Ingeniería de prompts: instrucción, contexto, rol y formato

**Duración sugerida:** 1.5 horas · **Entrega:** colección de prompts con entradas
de prueba, salidas esperadas y criterios de evaluación.

## Objetivos del laboratorio

- Diseñar prompts con **instrucción, contexto, rol y formato de salida**.
- Aplicar **restricciones** y **criterios de aceptación**.
- Usar **ejemplos *few-shot***.
- Producir **salidas estructuradas en JSON**.
- Practicar **clasificación, extracción y resumen**.

## Conexión con el proyecto ARAS

Los prompts de ARAS definen **cómo el sistema extrae, analiza y sintetiza**
información académica. Un buen prompt es medible y devuelve estructura estable,
lo que permite validar y encadenar pasos en el pipeline (Módulo 3).

## 1. Dependencias y helper

Creamos una función auxiliar que envía mensajes a Ollama y opcionalmente
fuerza salida JSON con `format="json"`.

In [ ]:
%pip install -q ollama

In [ ]:
import ollama
import json

MODELO = "llama3.2"

def preguntar(system, user, temperature=0.0, json_mode=False):
    kwargs = {
        "model": MODELO,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "options": {"temperature": temperature},
    }
    if json_mode:
        kwargs["format"] = "json"
    resp = ollama.chat(**kwargs)
    return resp["message"]["content"]

## 2. Los cuatro elementos de un prompt

Un prompt robusto combina:

1. **Rol:** quién es el modelo (ej. "eres un bibliotecario académico").
2. **Instrucción:** qué debe hacer.
3. **Contexto:** la información de entrada.
4. **Formato:** cómo debe devolver la respuesta.

Compara un prompt pobre con uno estructurado.

In [ ]:
# Prompt POBRE: ambiguo, sin rol ni formato.
pobre = "dime qué trata el texto"

# Prompt ESTRUCTURADO: rol + instrucción + contexto + formato.
sistema = "Eres un asistente académico. Responde solo lo que se pide, sin añadir opiniones."
usuario = (
    "Lee el siguiente texto y devuelve un resumen de una frase.\n\n"
    "TEXTO: El artículo describe cómo los asistentes basados en LLM reducen el "
    "tiempo de búsqueda documental en un 40%.\n\n"
    "FORMATO DE SALIDA: una única frase, sin comillas."
)

print("Prompt pobre:")
print(preguntar("Eres un asistente.", pobre))
print("\nPrompt estructurado:")
print(preguntar(sistema, usuario))

## 3. Restricciones y criterios de aceptación

Especificar **límites** (longitud, tono, idioma) y **criterios de aceptación**
(condiciones que la respuesta debe cumplir) hace el resultado verificable.

In [ ]:
sistema = "Eres un redactor técnico. Cumple exactamente las restricciones."
usuario = (
    "Explica qué es un 'agente de software'.\n\n"
    "RESTRICCIONES:\n"
    "- Máximo 30 palabras.\n"
    "- En español.\n"
    "- Sin tecnicismos innecesarios.\n\n"
    "CRITERIOS DE ACEPTACIÓN:\n"
    "- Debe contener la palabra 'autónomo'.\n"
    "- No debe mencionar 'robot'."
)
print(preguntar(sistema, usuario))

## 4. Ejemplos *few-shot*

Incluir 2–3 ejemplos de entrada → salida enseña al modelo el patrón esperado sin
necesidad de reentrenarlo.

In [ ]:
sistema = "Clasifica cada afirmación en: HECHO, OPINIÓN o HIPÓTESIS. Responde solo con la etiqueta."

ejemplos = (
    "EJEMPLOS:\n"
    "Entrada: 'La Tierra orbita alrededor del Sol.' -> HECHO\n"
    "Entrada: 'El cine clásico es mejor que el actual.' -> OPINIÓN\n"
    "Entrada: 'Quizá los agentes sustituyan a las interfaces actuales.' -> HIPÓTESIS\n\n"
)

entrada = "Entrada: 'Los modelos pequeños podrían superar a los grandes en tareas específicas.'"

print(preguntar(sistema, ejemplos + entrada, temperature=0.0))

## 5. Salidas estructuradas en JSON

Para que un pipeline consuma la respuesta, pedimos **JSON válido**. Ollama puede
forzar el formato con `format="json"` (lo que reduce, aunque no elimina, el riesgo
de JSON malformado).

In [ ]:
sistema = (
    "Eres un sistema de extracción de información. Devuelve SOLO un objeto JSON válido "
    "sin texto adicional, con las claves exactas: 'titulo' (string), 'autores' (lista de strings), "
    "'anio' (número) y 'hallazgo' (string)."
)
texto = (
    "El artículo 'Agentes conversacionales en el aula', publicado en 2024 por "
    "Laura Pérez y Juan Ríos, concluye que la retroalimentación inmediata mejora "
    "la retención en un 15%."
)
respuesta = preguntar(sistema, "TEXTO: " + texto, json_mode=True)
datos = json.loads(respuesta)
print(json.dumps(datos, indent=2, ensure_ascii=False))

## 6. Tareas típicas: clasificación, extracción y resumen

Reutilizamos el helper para tres tareas frecuentes en investigación.

In [ ]:
texto = (
    "El trabajo compara tres métodos de evaluación de agentes y halla que la "
    "evaluación basada en rúbricas correlaciona mejor con el juicio humano que "
    "las métricas automáticas."
)

tareas = {
    "clasificación": ("Clasifica el texto en una categoría: [evaluación, arquitectura, seguridad]. Responde solo la categoría.", False),
    "extracción": ("Extrae el objeto de estudio y el hallazgo principal como JSON con claves 'objeto' y 'hallazgo'.", True),
    "resumen": ("Resume el texto en una frase.", False),
}

for nombre, (instruccion, json_mode) in tareas.items():
    out = preguntar("Eres un analista académico. Responde solo lo pedido.", f"{instruccion}\n\nTEXTO: {texto}", json_mode=json_mode)
    print(f"--- {nombre} ---\n{out}\n")

## 7. Validación de salidas

Nunca confíes en que el JSON llega bien formado. Valida con un esquema y captura errores.

In [ ]:
import json

def extraer_seguro(texto, intentos=3):
    sistema = "Devuelve SOLO JSON con claves 'titulo' y 'autores'."
    for i in range(intentos):
        try:
            raw = preguntar(sistema, "TEXTO: " + texto, json_mode=True)
            datos = json.loads(raw)
            assert "titulo" in datos and "autores" in datos, "faltan claves"
            return datos
        except Exception as e:
            print(f"Intento {i+1} falló ({e}). Reintentando...")
    return None

resultado = extraer_seguro("El libro 'Sistemas Agénticos' fue escrito por Ana Ruiz.")
print(resultado)

## Resumen del laboratorio

- Diseñamos prompts con **rol + instrucción + contexto + formato**.
- Aplicamos **restricciones** y **criterios de aceptación**.
- Usamos **few-shot** y **salidas JSON** con validación.
- Practicamos **clasificación, extracción y resumen**.

**Entregable del Módulo 2:** colección de prompts documentados con entradas de
prueba, salidas esperadas y criterios de evaluación.